[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/04_agent_tools.ipynb) [![GitHub](https://img.shields.io/badge/GitHub-仓库主页-181717?logo=github)](https://github.com/Luca5702/langgraph-study)

**📚 课程导航：** [01](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/01_hello_graph.ipynb) · [02](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/02_state_reducers.ipynb) · [03](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/03_control_flow.ipynb) · **[04](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/04_agent_tools.ipynb)** · [05](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/05_persistence_memory.ipynb) · [06](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/06_human_in_the_loop.ipynb) · [07](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/07_time_travel.ipynb) · [08](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/08_streaming.ipynb) · [09](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/09_subgraphs.ipynb) · [10](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/10_functional_api.ipynb) · [11](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/11_production.ipynb)

[⬅️ 上一课：第 3 课：控制流（条件边 / 循环 / 并行 / Send / Command）](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/03_control_flow.ipynb) ｜ [➡️ 下一课：第 5 课：持久化与记忆（Checkpointer + Store）](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/05_persistence_memory.ipynb)

---

# 第 4 课：从零手写一个 ReAct Agent（LLM + 工具 + 循环）

Agent = 一个带循环的图：
  START -> llm --(有 tool_calls?)--> tools -> llm -> ... -> END
  \--(没有)----------------------------> END
* langgraph.prebuilt.ToolNode：执行 AIMessage 中的 tool_calls，返回 ToolMessage
* langgraph.prebuilt.tools_condition：有工具调用就去 "tools"，否则 END
* 生产中更常用 `langchain.agents.create_agent`（基于 LangGraph 构建），
  但亲手写一次能让你真正理解它在做什么。

In [1]:
#@title 🔧 第 0 步：初始化环境（在 Colab 中点击左侧 ▶ 即可自动安装依赖，本地运行自动跳过） { display-mode: "form" }
import importlib.util, subprocess, sys
if importlib.util.find_spec('langgraph') is None:
    print('⏳ 正在安装 LangGraph 环境（约 10 秒）...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'langgraph', 'langchain-core', 'langgraph-checkpoint-sqlite'])
    print('✅ 安装完成！')
else:
    import importlib.metadata as m
    print(f'✅ 环境已就绪：langgraph={m.version("langgraph")}')
from pathlib import Path
import urllib.request
for p in [Path('lessons'), Path.cwd().parent / 'lessons']:
    if (p / 'fake_llm.py').exists():
        sys.path.insert(0, str(p.resolve()))
        break
else:
    if not Path('fake_llm.py').exists():
        urllib.request.urlretrieve('https://raw.githubusercontent.com/Luca5702/langgraph-study/main/lessons/fake_llm.py', 'fake_llm.py')


✅ 环境已就绪：langgraph=1.2.22


In [2]:
from langchain_core.messages import SystemMessage
from langchain_core.tools import tool

from langgraph.graph import START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition

from fake_llm import get_model


@tool
def get_weather(city: str) -> str:
    """查询城市天气。"""
    fake = {"北京": "晴 12°C", "上海": "小雨 16°C", "深圳": "多云 24°C", "旧金山": "雾 14°C"}
    return f"{city}：{fake.get(city, '未知')}"


@tool
def calculator(expression: str) -> str:
    """计算一个算术表达式。"""
    return str(eval(expression, {"__builtins__": {}}))  # 仅为教学，生产勿用 eval


tools = [get_weather, calculator]
model = get_model().bind_tools(tools)


def llm(state: MessagesState):
    msgs = [SystemMessage("你是一个乐于助人的助手。")] + state["messages"]
    return {"messages": [model.invoke(msgs)]}


builder = StateGraph(MessagesState)
builder.add_node("llm", llm)
builder.add_node("tools", ToolNode(tools))
builder.add_edge(START, "llm")
builder.add_conditional_edges("llm", tools_condition)  # -> "tools" 或 END
builder.add_edge("tools", "llm")
agent = builder.compile()


def show(result):
    for m in result["messages"]:
        extra = f" tool_calls={m.tool_calls}" if getattr(m, "tool_calls", None) else ""
        print(f"  {type(m).__name__:12s} | {m.content}{extra}")

## ▶️ 运行实验

In [3]:
print(agent.get_graph().draw_mermaid())
for q in ["上海今天天气怎么样？", "帮我算一下 12*7+3", "你好"]:
    print(f"\n>>> {q}")
    show(agent.invoke({"messages": [{"role": "user", "content": q}]}))

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	llm(llm)
	tools(tools<hr/><small><em>__tools = get_weather, calculator</em></small>)
	__end__([<p>__end__</p>]):::last
	__start__ --> llm;
	llm -.-> __end__;
	llm -.-> tools;
	tools --> llm;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc


>>> 上海今天天气怎么样？
  HumanMessage | 上海今天天气怎么样？
  AIMessage    |  tool_calls=[{'name': 'get_weather', 'args': {'city': '上海'}, 'id': 'call_b6e9d7ea', 'type': 'tool_call'}]
  ToolMessage  | 上海：小雨 16°C
  AIMessage    | 根据工具「get_weather」的结果：上海：小雨 16°C。还有什么可以帮你？

>>> 帮我算一下 12*7+3
  HumanMessage | 帮我算一下 12*7+3
  AIMessage    |  tool_calls=[{'name': 'calculator', 'args': {'expression': '12*7+3'}, 'id': 'call_17642f88', 'type': 'tool_call'}]
  ToolMessage  | 87
  AIMessage    | 根据工具「calculator」的结果：87。还有什么可以帮你？

>>> 你好
  HumanMessage | 你好
  AIMessage    | （假模型）我收到了：你好。试试问我天气或算术题吧！


## ✏️ 练习：
 1. 新增一个工具 get_time(city)，并让假模型（fake_llm.py）识别 "几点"。
 2. 用 stream_mode="updates" 观察每一步是哪个节点产生了什么消息。
 3. 装上真模型（见 fake_llm.py 顶部说明），对比行为。

In [4]:
# 在这里写你的练习代码 👇


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/04_agent_tools.ipynb) [![GitHub](https://img.shields.io/badge/GitHub-仓库主页-181717?logo=github)](https://github.com/Luca5702/langgraph-study)

**📚 课程导航：** [01](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/01_hello_graph.ipynb) · [02](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/02_state_reducers.ipynb) · [03](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/03_control_flow.ipynb) · **[04](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/04_agent_tools.ipynb)** · [05](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/05_persistence_memory.ipynb) · [06](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/06_human_in_the_loop.ipynb) · [07](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/07_time_travel.ipynb) · [08](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/08_streaming.ipynb) · [09](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/09_subgraphs.ipynb) · [10](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/10_functional_api.ipynb) · [11](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/11_production.ipynb)

[⬅️ 上一课：第 3 课：控制流（条件边 / 循环 / 并行 / Send / Command）](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/03_control_flow.ipynb) ｜ [➡️ 下一课：第 5 课：持久化与记忆（Checkpointer + Store）](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/05_persistence_memory.ipynb)

---